In [2]:
from astropy.io import fits
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import  numpy as np
from scipy import stats

In [3]:
dark_names = [f'lm_260429_{i:06d}.fits.gz' for i in range(51, 101)] #love f strings
#i is he number (51, 52, 53...)
#:06d is format it as 6 digits with leading zeros

# checking names
print(dark_names[0])   
print(dark_names[-1])  
print(len(dark_names)) 


#other fits

grid_names = [f'lm_260429_{i:06d}.fits.gz'for i in range(1, 51)]
pseudo_flat_names = [f'lm_260429_{i:06d}.fits.gz' for i in range (101, 151)]
#chaking names 
print(grid_names[0])
print(grid_names[-1])
print(len(grid_names))
print(pseudo_flat_names[0])
print(pseudo_flat_names[-1])
print(len(pseudo_flat_names))

lm_260429_000051.fits.gz
lm_260429_000100.fits.gz
50
lm_260429_000001.fits.gz
lm_260429_000050.fits.gz
50
lm_260429_000101.fits.gz
lm_260429_000150.fits.gz
50


computing means and getting headers

In [6]:
first_dark = fits.open(dark_names[0])
mean_dark = first_dark[0].data.astype(np.float32)
dark_header = first_dark[0].header.copy()
first_dark.close()

first_grid =fits.open(grid_names[0])
mean_grid = first_grid[0].data.astype(np.float32)
grid_header = first_grid[0].header.copy()
first_grid.close()

first_pseudo =fits.open(pseudo_flat_names[0])
mean_pseudo = first_pseudo[0].data.astype(np.float32)
pseudo_header = first_pseudo[0].header.copy()
first_pseudo.close()

loop for averaging

In [8]:
for name in dark_names[1:]:
    f= fits.open(name)
    mean_dark += f[0].data.astype(np.float32)
    f.close() 

mean_dark /= len(dark_names) 

for name in grid_names[1:]:
    f =fits.open(name)
    mean_grid += f[0].data.astype(np.float32)
    f.close()
mean_grid /= len(grid_names)    
    
for name in pseudo_flat_names[1:]:
    f = fits.open(name)
    mean_pseudo += f[0].data.astype(np.float32)
    f.close()

mean_pseudo /= len(pseudo_flat_names)


print("dark")
print(mean_dark)
print("pseudo")
print(mean_pseudo)
print("grid") 
print(mean_grid)
  

dark
[[[ 679.4696   671.02     646.2444  ...  737.97925  663.3864   763.95844]
  [ 651.20435  608.53076  630.9592  ...  643.9392   678.98     732.57245]
  [ 619.00037  582.8368   628.8768  ...  645.4696   682.40875  761.     ]
  ...
  [ 575.1836   477.40802  481.73398 ...  581.97925  578.898    642.408  ]
  [ 525.2644   510.4888   516.4068  ...  605.42755  572.59076  603.8572 ]
  [4095.      1381.2036   114.53    ...  145.3268   197.51039  184.8768 ]]

 [[ 713.8576   705.5513   681.1428  ...  774.2456   698.28564  798.7551 ]
  [ 685.7556   643.2452   665.1436  ...  678.8576   713.6332   768.4304 ]
  [ 652.9184   617.20435  663.26483 ...  680.368    717.7556   795.5112 ]
  ...
  [ 701.2864   604.65283  609.8564  ...  711.776    708.164    772.34717]
  [ 651.6112   638.46924  644.1012  ...  734.77515  702.08154  732.858  ]
  [4095.      1635.2728   247.69481 ...  272.082    325.0008   311.5108 ]]]
pseudo
[[[ 679.99963  671.36755  646.5304  ...  738.428    663.5508   763.8572 ]
  [ 651.36

create normalized flat 

In [9]:
superflat_sub = mean_pseudo - mean_dark
superflat_normalized = superflat_sub / np.median(superflat_sub)

print("Median:", np.median(superflat_normalized))  
print("Min:", superflat_normalized.min())
print("Max:", superflat_normalized.max())


Median: 1.0
Min: -1.984564
Max: 17.689793


saving files

In [10]:
dark_header['COMMENT'] = 'Superdark: average of 50 dark frames'
fits.PrimaryHDU(mean_dark, header=dark_header).writeto('superdark.fits', overwrite=True)

pseudo_header['COMMENT'] = 'Superflat: average of 50 pseudo-flat frames dark subtracted normalized'
fits.PrimaryHDU(superflat_normalized, header=pseudo_header).writeto('superflat_normalized.fits', overwrite=True)

print("Saved!")


Saved!


correcting grids

In [11]:
for i, name in enumerate(grid_names):
    f = fits.open(name)
    grid_frame = f[0].data.astype(np.float32)
    grid_frame_header = f[0].header.copy()
    f.close()
    
corrected = (grid_frame - mean_dark) / superflat_normalized  

grid_frame_header['COMMENT'] = 'Dark subtracted and flat field corrected'
outname = f'corrected_{i+1:03d}.fits'
fits.PrimaryHDU(corrected, header=grid_frame_header).writeto(outname, overwrite=True)
    
print(f"Saved {outname}")

print(" 50 grid frames corrected")  


Saved corrected_050.fits
 50 grid frames corrected


/var/folders/_q/b3__rd897hx0zppc2zqjkln40000gp/T/ipykernel_8444/2473882574.py:7: RuntimeWarning: divide by zero encountered in divide
  corrected = (grid_frame - mean_dark) / superflat_normalized
/var/folders/_q/b3__rd897hx0zppc2zqjkln40000gp/T/ipykernel_8444/2473882574.py:7: RuntimeWarning: invalid value encountered in divide
  corrected = (grid_frame - mean_dark) / superflat_normalized
